## Prueba MANUAL de RUTINAS


In [1]:
import psycopg2 as pg2
from datetime import date
import time
import logging
import os
import sys
from dotenv import dotenv_values
import pandas as pd
import warnings
# Configuración global
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

# Configurar logging
logging.basicConfig(
    filename='./logs/monitoreo_tablas.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

ENV_PATH = os.environ.get("ETL_ENV_PATH", "E:/ETL/ETL_DIARCO/.env")  # Toma Producción si está definido, o la ruta por defecto E:\ETL\ETL_DIARCO\.env
# Verificar si el archivo .env existe
if not os.path.exists(ENV_PATH):
    print(f"El archivo .env no existe en la ruta: {ENV_PATH}")
    print(f"Directorio actual: {os.getcwd()}")
    sys.exit(1)
    
secrets = dotenv_values(ENV_PATH)

def Open_Diarco_Data(): 
    conn_str = f"dbname={secrets['PG_DB']} user={secrets['PG_USER']} password={secrets['PG_PASSWORD']} host={secrets['PG_HOST']} port={secrets['PG_PORT']}"
    #print (conn_str)
    for i in range(5):
        try:    
            conn = pg2.connect(conn_str)
            return conn
        except Exception as e:
            print(f'Error en la conexión: {e}')
            time.sleep(5)
    return None  # Retorna None si todos los intentos fallan


### GENERAR datos Validar VENTAS en CERO

In [2]:
from datetime import datetime, timedelta
import sqlalchemy
import matplotlib.pyplot as plt 
import numpy as np
import time

def Open_Connexa_Alquemy():
    DB_TYPE = "postgresql"
    DB_USER = secrets['PGP_USER']
    DB_PASS = secrets['PGP_PASSWORD']  # ⚠️ Reemplazar por la contraseña real o usar variables de entorno
    DB_HOST = secrets['PGP_HOST']
    DB_PORT = secrets['PGP_PORT']
    DB_NAME = secrets['PGP_DB']

    # Crear engine de conexión
    try:
        engine = sqlalchemy.create_engine(
        f"{DB_TYPE}://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
        )
        conn = engine.connect()
        return conn
    except Exception as e:
        print(f'Error en la conexión: {e}')
        return None   
    
def Open_Conn_Postgres():
    conn_str = f"dbname={secrets['PGP_DB']} user={secrets['PGP_USER']} password={secrets['PGP_PASSWORD']} host={secrets['PGP_HOST']} port={secrets['PGP_PORT']}"
    for i in range(5):
        try:
            conn = pg2.connect(conn_str)
            return conn 
        except Exception as e:
            print(f"Error en la conexión, intento {i+1}/{5}: {e}")
            time.sleep(5)
    return None  # Retorna None si todos los intentos fallan

def Close_Connection(conn): 
    if conn is not None:
        conn.close()
        # print("✅ Conexión cerrada.")    
    return True

In [3]:
# Nueva Rutina al MIGRAR a PostgreSQL y Ejecución REMOTA
# 2025-05-16 Se agrega c_comprador
def generar_datos(id_proveedor, etiqueta, ventana):
    folder = secrets["FOLDER_DATOS"]
    archivo_datos = f'{folder}/{etiqueta}.csv'
    archivo_articulos = f'{folder}/{etiqueta}_articulos.csv'
    archivo_stock = f'{folder}/{etiqueta}_stock_sucursal.csv'
    
    # Verificar la fecha de modificación del archivo
    if os.path.exists(archivo_datos):
        fecha_modificacion = datetime.fromtimestamp(os.path.getmtime(archivo_datos))
        if fecha_modificacion.date() == datetime.today().date():
            try:
                data = pd.read_csv(archivo_datos)
                data = data.dropna(subset=['Codigo_Articulo', 'Sucursal', 'Fecha'], how='all')  # Eliminar filas donde todas las columnas clave son NaN
                data['Codigo_Articulo'] = data['Codigo_Articulo'].astype(int)
                data['Sucursal'] = data['Sucursal'].astype(int)
                data['Fecha'] = pd.to_datetime(data['Fecha'])

                articulos = pd.read_csv(archivo_articulos)
                articulos = articulos.dropna(subset=['C_ARTICULO', 'C_SUCU_EMPR'], how='all')  # Eliminar filas donde todas las columnas clave son NaN
                

                print(f"-> Datos Recuperados del CACHE: {id_proveedor}, Label: {etiqueta}")
                return data, articulos
            except Exception as e:
                print(f"Error al leer los archivos cacheados: {e}")
        else:
            # Eliminar archivos si la fecha no es la de hoy
            os.remove(archivo_datos)
            if os.path.exists(archivo_articulos):
                os.remove(archivo_articulos)
            print(f"-> Archivos eliminados por ser obsoletos: {archivo_datos}, {archivo_articulos}")
    else:
        print(f"-> Generando datos para ID: {id_proveedor}, Label: {etiqueta}")
        # Aquí puedes incluir el código para generar los datos si no EXISTE el Archivo en el CACHE
        conn = Open_Diarco_Data()

        # --- ARTÍCULOS --- NUEVA FUENTE GLOBAL 06/25 -- SP_BASE_PRODUCTOS_VIGENTES
        query_articulos = f"""
            SELECT DISTINCT c_sucu_empr, c_articulo, c_proveedor_primario, abastecimiento, cod_cd, habilitado,  
                    cod_comprador AS c_comprador, 
                    q_factor_compra, full_capacity_pallet, number_of_layers, number_of_boxes_per_layer
            FROM src.base_productos_vigentes
            WHERE c_proveedor_primario = {id_proveedor}
            ORDER BY c_articulo, c_proveedor_primario;
        """
        articulos = pd.read_sql(query_articulos, conn) # type: ignore
        if articulos.empty:
            print(f"❗ No se encontraron artículos para el proveedor {id_proveedor}.")
            Close_Connection(conn)
            return None, None

        articulos.columns = articulos.columns.str.upper()
        articulos.rename(columns={'COD_COMPRADOR': 'C_COMPRADOR'}, inplace=True)   # En la base nueva se llama distinto
        articulos['C_SUCU_EMPR'] = articulos['C_SUCU_EMPR'].astype(int)
        articulos['C_ARTICULO'] = articulos['C_ARTICULO'].astype(int)
        articulos['C_PROVEEDOR_PRIMARIO'] = articulos['C_PROVEEDOR_PRIMARIO'].astype(int)
        articulos['ABASTECIMIENTO'] = articulos['ABASTECIMIENTO'].astype(int)
        articulos['HABILITADO'] = articulos['HABILITADO'].astype(int)
        articulos['C_COMPRADOR'] = articulos['C_COMPRADOR'].astype(int)
        articulos['Q_FACTOR_COMPRA'] = articulos['Q_FACTOR_COMPRA'].astype(int)
        articulos['FULL_CAPACITY_PALLET'] = articulos['FULL_CAPACITY_PALLET'].astype(int)
        articulos['NUMBER_OF_LAYERS'] = articulos['NUMBER_OF_LAYERS'].astype(int)
        articulos['NUMBER_OF_BOXES_PER_LAYER'] = articulos['NUMBER_OF_BOXES_PER_LAYER'].astype(int)
        articulos.to_csv(archivo_articulos, index=False, encoding='utf-8')
        print(f"---> Datos de Artículos guardados")        
        
        # --- BASE STOCK --- NUEVA FUENTE GLOBAL 06/25 -- SP_BASE_STOCK_SUCURSAL
        query_stock_sucursal = f"""
            SELECT codigo_articulo, codigo_sucursal, codigo_proveedor, pedido_sgm, stock, 
                pedido_pendiente, i_lista_calculado, factor_venta, precio_venta, precio_costo, 
                q_dias_stock, transfer_pendiente, venta_unidades_1q, venta_unidades_2q
            FROM src.base_stock_sucursal
            WHERE codigo_proveedor = {id_proveedor}
            ORDER BY codigo_articulo, codigo_articulo;
        """
        stock_sucursal = pd.read_sql(query_stock_sucursal, conn) # type: ignore
        if stock_sucursal.empty:
            print(f"❗ No se encontraron artículos de Stock_Sucursal para el proveedor {id_proveedor}.")
            Close_Connection(conn)
            return None, None
        
        # Limpieza general antes de conversión
        stock_sucursal = stock_sucursal.replace([np.inf, -np.inf], np.nan)
        stock_sucursal = stock_sucursal.fillna(0)

        stock_sucursal.columns = stock_sucursal.columns.str.upper()
        #  Cambiar tipos de datos
        stock_sucursal['CODIGO_SUCURSAL'] = stock_sucursal['CODIGO_SUCURSAL'].astype(int)
        stock_sucursal['CODIGO_ARTICULO'] = stock_sucursal['CODIGO_ARTICULO'].astype(int)
        stock_sucursal['CODIGO_PROVEEDOR'] = stock_sucursal['CODIGO_PROVEEDOR'].astype(int)
        stock_sucursal["PEDIDO_SGM"] = pd.to_numeric(stock_sucursal["PEDIDO_SGM"], errors="coerce").astype("Float64")
        stock_sucursal["STOCK"] = pd.to_numeric(stock_sucursal["STOCK"], errors="coerce").astype("Float64")
        stock_sucursal["PEDIDO_PENDIENTE"] = pd.to_numeric(stock_sucursal["PEDIDO_PENDIENTE"], errors="coerce").astype("Float64")
        stock_sucursal["I_LISTA_CALCULADO"] = pd.to_numeric(stock_sucursal["I_LISTA_CALCULADO"], errors="coerce").astype("Float64")
        stock_sucursal['FACTOR_VENTA'] = stock_sucursal['FACTOR_VENTA'].astype(int)
        stock_sucursal['PRECIO_VENTA'] = pd.to_numeric(stock_sucursal['PRECIO_VENTA'], errors='coerce').astype('Float64')
        stock_sucursal['PRECIO_COSTO'] = pd.to_numeric(stock_sucursal['PRECIO_COSTO'], errors='coerce').astype('Float64')
        stock_sucursal['Q_DIAS_STOCK'] = pd.to_numeric(stock_sucursal['Q_DIAS_STOCK'], errors='coerce').astype('Int64')
        stock_sucursal['TRANSFER_PENDIENTE'] = pd.to_numeric(stock_sucursal['TRANSFER_PENDIENTE'], errors='coerce').astype('Float64')
        stock_sucursal['VENTA_UNIDADES_1Q'] = pd.to_numeric(stock_sucursal['VENTA_UNIDADES_1Q'], errors='coerce').astype('Float64')
        stock_sucursal['VENTA_UNIDADES_2Q'] = pd.to_numeric(stock_sucursal['VENTA_UNIDADES_2Q'], errors='coerce').astype('Float64')

        stock_sucursal.to_csv(archivo_stock, index=False, encoding='utf-8')
        print(f"---> Datos de Stock Sucursal guardados")
        
        # -- COMBINAR ARTÍCULOS y STOCK --
        articulos = pd.merge(articulos, stock_sucursal, left_on=['C_ARTICULO', 'C_SUCU_EMPR'], right_on=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], how='inner')  

        # --- VENTAS --- DIARCO + BARRIO ( En 2 Bases de Datos distintas )
        query_ventas = f"""
            SELECT 
                v.f_venta AS Fecha, 
                v.c_articulo AS Codigo_Articulo, 
                v.c_sucu_empr AS Sucursal, 
                v.q_unidades_vendidas AS Unidades
            FROM src.t702_est_vtas_por_articulo v
            JOIN src.base_productos_vigentes a 
                ON a.c_articulo = v.c_articulo
                AND a.c_sucu_empr = v.c_sucu_empr
                AND a.c_proveedor_primario = {id_proveedor}
            WHERE v.f_venta >= '2024-06-01'

            UNION ALL

            SELECT 
                v.f_venta AS Fecha, 
                v.c_articulo AS Codigo_Articulo, 
                v.c_sucu_empr AS Sucursal, 
                v.q_unidades_vendidas AS Unidades
            FROM src.t702_est_vtas_por_articulo_dbarrio v
            JOIN src.base_productos_vigentes a 
                ON a.c_articulo = v.c_articulo
                AND a.c_sucu_empr = v.c_sucu_empr
                AND a.c_proveedor_primario = {id_proveedor}
            WHERE v.f_venta >= '2024-06-01'

            ORDER BY Fecha, Codigo_Articulo, Sucursal;
        """
        ventas = pd.read_sql(query_ventas, conn) # type: ignore
        if ventas.empty:
            print(f"⚠️ No se encontraron ventas DIARCO + BARRIO para el proveedor {id_proveedor}.")
        
        # Convertir columnas a minúsculas si hay datos
        if not ventas.empty:
            ventas.columns = ventas.columns.str.lower()
            # Transformar tipos de datos si hay datos
            ventas['sucursal'] = ventas['sucursal'].astype(int)
            ventas['codigo_articulo'] = ventas['codigo_articulo'].astype(int)
            ventas['fecha'] = pd.to_datetime(ventas['fecha'])
            # Eliminar filas con NaN en 'fecha', 'codigo_articulo' o 'sucursal'
            ventas = ventas.dropna(subset=['fecha', 'codigo_articulo', 'sucursal'], how='all')
            
        else:
            print(f"⚠️ No se encontraron ventas para el proveedor {id_proveedor} ni en DIARCO ni en BARRIO.")
            ventas = pd.DataFrame(columns=['fecha', 'codigo_articulo', 'sucursal', 'unidades'])  # DataFrame vacío con columnas esperadas

        ventas = ventas.rename(columns={
            "fecha": "Fecha",
            "codigo_articulo": "Codigo_Articulo",
            "sucursal": "Sucursal",
            "unidades": "Unidades"
        })
        # Guardar solo VENTAS 
        ventas.to_csv(f'{folder}/{etiqueta}_Demanda.csv', index=False, encoding='utf-8')
        print(f"---> Datos de Ventas guardados")

        # --- MERGE ---
        data = pd.merge(
            articulos,
            ventas,  
            left_on=['C_ARTICULO', 'C_SUCU_EMPR'],          
            right_on=['Codigo_Articulo', 'Sucursal'],            
            how='left'   # 'inner'  # Solo traer los productos que están en AMBOS ARCHIVOS  'left' trate TODOS los articulos activos en cada sucursal
        )

        if data.empty:
            print(f"⚠️ No hay coincidencias entre artículos y ventas para el proveedor {id_proveedor}.")
            Close_Connection(conn)
            return None, articulos

        # Conversión segura de columnas a enteros con soporte para NaN
        data['C_ARTICULO'] = data['C_ARTICULO'].astype('Int64')
        data['C_SUCU_EMPR'] = data['C_SUCU_EMPR'].astype('Int64')
        data['Codigo_Articulo'] = data['Codigo_Articulo'].astype('Int64')
        data['Sucursal'] = data['Sucursal'].astype('Int64')

        # Agregar columna indicadora de si tiene demanda
        data['TIENE_DEMANDA'] = data['Unidades'].notna().astype(int)
        data['Unidades'] = data['Unidades'].fillna(0)

        # Guardado
        data.to_csv(archivo_datos, index=False, encoding='utf-8')
        print(f"---> Datos de RECUPERACIÓN guardados")
        
        # Guardar los datos Compactos de VENTAS en un archivo CSV con el nombre del Proveedor y sufijo _Ventas
        file_path = f'{folder}/{etiqueta}_Ventas.csv'
        print(f"[DEBUG] Ruta destino definida en .env: {folder}")

        # Eliminar Columnas Innecesarias
        data = data[['Fecha', 'Codigo_Articulo', 'Sucursal', 'Unidades']]
        data.to_csv(file_path, index=False, encoding='utf-8')
        print(f"---> Datos de Ventas guardados: {file_path}")  

        Close_Connection(conn)
        return data, articulos

In [4]:
# Configurar Variables

id_proveedor = 6144
etiqueta ='CLEANER'
ventana = 30  # Ventana de días para la demanda
algoritmo = 'ALGO_07'  # Ejemplo de algoritmo, puedes cambiarlo según sea necesario

datos, articulos = generar_datos(id_proveedor, etiqueta, ventana)

-> Generando datos para ID: 6144, Label: CLEANER
---> Datos de Artículos guardados
---> Datos de Stock Sucursal guardados
---> Datos de Ventas guardados
---> Datos de RECUPERACIÓN guardados
[DEBUG] Ruta destino definida en .env: data
---> Datos de Ventas guardados: data/CLEANER_Ventas.csv


In [5]:
folder = secrets["FOLDER_DATOS"]

In [ ]:
### APLICAR ALGORITMO

###----------------------------------------------------------------
# ALGO_07 Demanda Simple x Factor -  Fecha de Base Movil
###----------------------------------------------------------------
def Calcular_Demanda_ALGO_07(df, id_proveedor, etiqueta, periodo, current_date,  factor, fecha_base):
    print('Dentro del Calcular_Demanda_ALGO_07')
    print(f'FORECAST control: {id_proveedor} - {etiqueta} - ventana: {periodo} - Fecha Actual: {current_date} - factor: {factor}  - Fecha de Base: {fecha_base}')
    
    start_time = time.time()
    
    # Convertir Parámetros a INT o FLOAT
    period_length = int(periodo)  # Asegurarse de que sea un entero
    base_date = pd.to_datetime(fecha_base, errors='coerce')  # Convertir a datetime, manejar errores
    factor = float(factor)

    # Convertir la columna 'Fecha' a tipo datetime si no lo está
    if not pd.api.types.is_datetime64_any_dtype(df['Fecha']):
        df['Fecha'] = pd.to_datetime(df['Fecha'], errors='coerce')
        df.dropna(subset=['Fecha'], inplace=True)  # Eliminar filas con fechas inválidas
    
    # Definir rango de fechas
    last_period_start = base_date     
    last_period_end = base_date + pd.Timedelta(days=period_length)

    # Filtrar los datos para cada uno de los períodos
    df_last = df[(df['Fecha'] >= last_period_start) & (df['Fecha'] <= last_period_end)]

    # Agregar las ventas (unidades) por artículo y sucursal para cada período
    sales_last = df_last.groupby(['Codigo_Articulo', 'Sucursal'])['Unidades'] \
                        .sum().reset_index().rename(columns={'Unidades': 'ventas_last'})

    # Unir la información de los tres períodos
    df_forecast = sales_last.copy() 
    df_forecast.fillna(0, inplace=True)

    # Calcular la demanda estimada como el promedio de las ventas del período multiplicado pro el factor
    df_forecast['Forecast'] = (df_forecast['ventas_last'] * factor)       #Aplico el peso absoluto de los factores.
    
    elapsed = round(time.time() - start_time, 2)
    print(f"🖼️ Preparación de Datos - Tiempo: {elapsed} seg")
    # Redondear la predicción al entero más cercano  y eliminar los Negativos
    df_forecast['Forecast'] = np.ceil(df_forecast['Forecast']).clip(lower=0) # type: ignore
    df_forecast['Average'] = round(df_forecast['Forecast'] /period_length ,3)
    
    # Agregar las columnas id_proveedor y ventana
    df_forecast['id_proveedor'] = id_proveedor
    df_forecast['algoritmo'] = 'ALGO_07'
    df_forecast['ventana'] = period_length
    df_forecast['f1'] = factor
    df_forecast['f2'] = 'na'
    df_forecast['f3'] = 'na'
    df_forecast['ventas_previous'] = 0  # Por compatibilidad con la estructura
    df_forecast['ventas_same_year'] = 0
    df_forecast['Fecha_Pronostico'] = base_date 

    # Reordenar las columnas según la especificación
    df_forecast = df_forecast[['id_proveedor', 'Codigo_Articulo', 'Sucursal',  'algoritmo', 'ventana', 'f1', 'f2', 'f3', 'Fecha_Pronostico',
                            'Forecast', 'Average','ventas_last', 'ventas_previous', 'ventas_same_year']]
    
    elapsed = round(time.time() - start_time, 2)
    print(f"🖼️ Demanda Calculada - Tiempo: {elapsed} seg")
    return df_forecast


    # Borrar Columnas Innecesarias
    # forecast_df.drop(columns=['ventas_last', 'ventas_previous', 'ventas_same_year'], inplace=True)

###----------------------------------------------------------------
# RUTINAS DE PROCESAMIENTO DE ALGORITMOS
###----------------------------------------------------------------

def Procesar_ALGO_07(data, articulos, proveedor, etiqueta, periodo, current_date, factor, fecha_base):    
    # Asignar valores por defecto si los factores no están definidos
    factor = 1 if factor is None else factor

    print(f'--> Procesar_ALGO_07 Período {periodo} - Factores Utilizados: Factor: {factor} Fecha Base: {fecha_base}')
        
    df_forecast = Calcular_Demanda_ALGO_07(data, proveedor, etiqueta,  periodo, current_date, factor, fecha_base)
    df_forecast['Codigo_Articulo']= df_forecast['Codigo_Articulo'].astype(int)
    df_forecast['Sucursal']= df_forecast['Sucursal'].astype(int)
    
    surtido = articulos[['C_ARTICULO', 'C_SUCU_EMPR']]
    surtido = surtido.rename(columns={'C_ARTICULO': 'Codigo_Articulo', 'C_SUCU_EMPR': 'Sucursal'})
    surtido['Codigo_Articulo']= surtido['Codigo_Articulo'].astype(int)
    surtido['Sucursal']= surtido['Sucursal'].astype(int)
    
    df_final = pd.merge(surtido, df_forecast, on=['Codigo_Articulo', 'Sucursal'], how='left')  # Asegurar que todos los artículos del surtido estén presentes
    
    df_final.to_csv(f'{folder}/{etiqueta}_ALGO_07_Solicitudes_Compra.csv', index=False)   # Exportar el resultado a un CSV para su posterior procesamiento
    
    # Exportar_Pronostico(df_forecast, proveedor, etiqueta, 'ALGO_07')  # Impactar Datos en la Interface        
    return  

In [ ]:
# Ejemplo de uso
folder = secrets["FOLDER_DATOS"]
periodo = 30  # Ventana de días para la demanda
current_date = datetime.today()
fecha_base = '2024-06-01'  # Fecha base para el pronóstico
factor = 1 

print(f'--> Procesar_ALGO_07 Período {periodo} - Factores Utilizados: Factor: {factor} Fecha Base: {fecha_base}')
    
df_forecast = Calcular_Demanda_ALGO_07(datos, id_proveedor, etiqueta,  periodo, current_date, factor, fecha_base)
df_forecast['Codigo_Articulo']= df_forecast['Codigo_Articulo'].astype(int)
df_forecast['Sucursal']= df_forecast['Sucursal'].astype(int)
df_forecast.to_csv(f'{folder}/{etiqueta}_ALGO_07_Solicitudes_Compra.csv', index=False)   # Exportar el resultado a un CSV para su posterior procesamiento


--> Procesar_ALGO_07 Período 30 - Factores Utilizados: Factor: 1 Fecha Base: 2024-06-01
Dentro del Calcular_Demanda_ALGO_07
FORECAST control: 6144 - CLEANER - ventana: 30 - Fecha Actual: 2025-07-26 13:15:55.263757 - factor: 1  - Fecha de Base: 2024-06-01
🖼️ Preparación de Datos - Tiempo: 0.0 seg
🖼️ Demanda Calculada - Tiempo: 0.01 seg


In [ ]:
folder = secrets["FOLDER_DATOS"]

df_ventas = pd.read_csv(f'{folder}/{etiqueta}_Ventas.csv')

# Convertir tipos de datos    
df_ventas['Codigo_Articulo'] = pd.to_numeric(df_ventas['Codigo_Articulo'], errors='coerce').astype('Int64')
df_ventas['Sucursal'] = pd.to_numeric(df_ventas['Sucursal'], errors='coerce').astype('Int64')   
df_ventas['Fecha']= pd.to_datetime(df_ventas['Fecha'])

df_ventas = df_ventas.dropna(subset=['Codigo_Articulo', 'Sucursal'], how='all')  # Eliminar filas donde ambas columnas son NaN

if df_ventas[['Codigo_Articulo', 'Sucursal']].isnull().any().any():
    print(f"⚠️ Atención: Ventas contiene registros con valores nulos en Código o Sucursal.")

In [ ]:
## EXTENDER DATOS
conn = Open_Conn_Postgres()

# Obtener Sites
stores = pd.read_sql("SELECT code, name, id FROM public.fnd_site", conn) # type: ignore
stores = stores[pd.to_numeric(stores['code'], errors='coerce').notna()].copy()
stores['code'] = stores['code'].astype(int)

# Obtener Productos
products = pd.read_sql("SELECT ext_code, description, id FROM public.fnd_product", conn) # type: ignore
products = products[pd.to_numeric(products['ext_code'], errors='coerce').notna()].copy()
products['ext_code'] = products['ext_code'].astype(int)
assert products['ext_code'].is_unique, "ERROR: productos.ext_code tiene duplicados"


Close_Connection(conn)

# Unir con productos y validar
df_merged = df_forecast.merge(products, left_on='Codigo_Articulo', right_on='ext_code', how='left')
df_merged.rename(columns={'id': 'product_id'}, inplace=True)
df_merged.drop(columns=['ext_code', 'description'], inplace=True)

# Unir con sites y validar
df_merged = df_merged.merge(stores, left_on='Sucursal', right_on='code', how='left')
df_merged.rename(columns={'id': 'site_id'}, inplace=True)
df_merged.drop(columns=['code', 'name'], inplace=True)

# Validación de integridad referencial
errores = df_merged[df_merged['site_id'].isna() | df_merged['product_id'].isna()]
if not errores.empty:
    print(f"❌ Error: Se encontraron {len(errores)} registros con site_id o product_id nulos.")
    errores[['Codigo_Articulo', 'Sucursal']].drop_duplicates().to_csv(
        f"{folder}/{algoritmo}_Errores_Missing_UUID.csv", index=False)
    raise ValueError("Existen artículos o sucursales no presentes en Connexa. Revisión necesaria.")


df_nuevo = articulos.copy()   # Articulos ya tiene SP_BASE_ARTICULOS_SUCURSAL
# -- COMBINAR ARTÍCULOS y STOCK --
df_nuevo = pd.merge(df_nuevo, stock_sucursal, left_on=['C_ARTICULO', 'C_SUCU_EMPR'], right_on=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], how='inner')
df_nuevo.drop(columns=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], inplace=True) 
df_nuevo['C_SUCU_EMPR'] = df_nuevo['C_SUCU_EMPR'].astype(int)
df_nuevo['C_ARTICULO'] = df_nuevo['C_ARTICULO'].astype(int)

duplicados = df_nuevo[df_nuevo.duplicated(['C_SUCU_EMPR', 'C_ARTICULO'], keep=False)]
if not duplicados.empty:
    print(f"🚨 Hay {len(duplicados)} filas duplicadas en df_nuevo por artículo+sucursal")
    duplicados.to_csv(f"{folder}/Duplicados_df_nuevo.csv", index=False)

    #Elimino Duplicados
    df_nuevo = df_nuevo.drop_duplicates(subset=['C_SUCU_EMPR', 'C_ARTICULO'])

df_merged = df_merged.merge(
    df_nuevo,
    left_on=['Sucursal', 'Codigo_Articulo'],
    right_on=['C_SUCU_EMPR', 'C_ARTICULO'],
    how='left'
)
df_merged.drop(columns=['C_SUCU_EMPR', 'C_ARTICULO'], inplace=True)

print(f"🔍 Forecast original: {len(df_forecast)} registros")
print(f"📈 Forecast extendido: {len(df_merged)} registros")





In [ ]:
# Punto de entrada
folder = secrets["FOLDER_DATOS"]

try:
    df_extendido = extender_datos_forecast(algoritmo, etiqueta, id_proveedor)

    # Guardar archivo extendido
    file_path = f"{folder}/{algoritmo}_Pronostico_Extendido.csv"
    df_extendido.to_csv(file_path, index=False)
    print(f"✅ Archivo guardado: {file_path}")


except ValueError as ve:
    print(f"❌ Error VALIDACIÓN {etiqueta}: {ve}")

except Exception as e:
    print(f"❌ Error procesando {etiqueta}: {e}")